# Phase 4/5/6 — Model, Training & Evaluation

Covers:
- NRMS model architecture (NewsEncoder, UserEncoder)
- PyTorch Dataset and DataLoader
- Training loop with loss curves
- Hyperparameter experiments
- Evaluation: AUC, MRR, nDCG@5, nDCG@10
- Error analysis

In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import os
import random
import time
import sys
sys.path.insert(0, '../src')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

os.makedirs('models',  exist_ok=True)
os.makedirs('results', exist_ok=True)

## 1. Load Processed Data

In [ ]:
DATA_DIR = 'data/processed'

with open(f'{DATA_DIR}/train_samples.pkl', 'rb') as f:
    train_samples = pickle.load(f)

with open(f'{DATA_DIR}/dev_samples.pkl', 'rb') as f:
    dev_samples = pickle.load(f)

embedding_matrix = np.load(f'{DATA_DIR}/embedding_matrix.npy')

print(f'Train samples    : {len(train_samples):,}')
print(f'Dev samples      : {len(dev_samples):,}')
print(f'Embedding matrix : {embedding_matrix.shape}')

## 2. PyTorch Dataset

In [ ]:
class MINDTrainDataset(Dataset):
    """Fixed-size candidates (1 pos + NEG_K neg). Used for training."""
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            'history'   : torch.tensor(s['history'],    dtype=torch.long),
            'hist_mask' : torch.tensor(s['hist_mask'],  dtype=torch.long),
            'candidates': torch.tensor(s['candidates'], dtype=torch.long),
            # label for CrossEntropyLoss: index of positive = 0
            'label'     : torch.tensor(0,               dtype=torch.long),
        }


train_dataset = MINDTrainDataset(train_samples)
print(f'Dataset size: {len(train_dataset):,}')

# Verify a batch
sample_item = train_dataset[0]
print('history shape   :', sample_item['history'].shape)
print('candidates shape:', sample_item['candidates'].shape)

## 3. Model Architecture

In [ ]:
class AdditiveAttention(nn.Module):
    """
    Collapses a sequence (batch, seq, dim) into a single vector (batch, dim)
    using a learned additive attention query.
    """
    def __init__(self, dim, hidden_dim=200):
        super().__init__()
        self.proj  = nn.Linear(dim, hidden_dim)
        self.query = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, x, mask=None):
        # x: (B, T, D)
        e      = torch.tanh(self.proj(x))          # (B, T, H)
        scores = self.query(e).squeeze(-1)          # (B, T)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        weights = F.softmax(scores, dim=-1)         # (B, T)
        out     = torch.bmm(weights.unsqueeze(1), x).squeeze(1)  # (B, D)
        return out


class NewsEncoder(nn.Module):
    """
    Encodes a news title (sequence of word ids) into a fixed-size vector.
    Pipeline: Embedding → Projection → Multi-Head Self-Attention → Additive Attention
    """
    def __init__(self, embedding_matrix, num_heads=16, head_dim=16, dropout=0.2):
        super().__init__()
        embed_dim = embedding_matrix.shape[1]
        attn_dim  = num_heads * head_dim

        self.word_embed   = nn.Embedding.from_pretrained(
            torch.FloatTensor(embedding_matrix), freeze=False, padding_idx=0)
        self.proj         = nn.Linear(embed_dim, attn_dim)
        self.mha          = nn.MultiheadAttention(
            embed_dim=attn_dim, num_heads=num_heads, dropout=dropout,
            batch_first=True)
        self.additive_attn = AdditiveAttention(attn_dim)
        self.dropout      = nn.Dropout(dropout)
        self.norm         = nn.LayerNorm(attn_dim)

    def forward(self, title_ids):
        # title_ids: (B, T)
        pad_mask = (title_ids == 0)                         # True where padding
        x = self.dropout(self.word_embed(title_ids))        # (B, T, E)
        x = self.proj(x)                                    # (B, T, D)
        x, _ = self.mha(x, x, x, key_padding_mask=pad_mask)
        x = self.norm(self.dropout(x))
        news_vec = self.additive_attn(x)                    # (B, D)
        return news_vec


class UserEncoder(nn.Module):
    """
    Aggregates a sequence of clicked-news vectors into a user representation.
    Pipeline: Multi-Head Self-Attention over history → Additive Attention
    """
    def __init__(self, news_dim, num_heads=16, dropout=0.2):
        super().__init__()
        self.mha           = nn.MultiheadAttention(
            embed_dim=news_dim, num_heads=num_heads, dropout=dropout,
            batch_first=True)
        self.additive_attn = AdditiveAttention(news_dim)
        self.dropout       = nn.Dropout(dropout)
        self.norm          = nn.LayerNorm(news_dim)

    def forward(self, hist_vecs, hist_mask=None):
        # hist_vecs: (B, H, D); hist_mask: (B, H) — 1 = real, 0 = pad
        key_pad = (hist_mask == 0) if hist_mask is not None else None
        x, _ = self.mha(hist_vecs, hist_vecs, hist_vecs,
                        key_padding_mask=key_pad)
        x = self.norm(self.dropout(x))
        user_vec = self.additive_attn(x, hist_mask)         # (B, D)
        return user_vec


class NRMSModel(nn.Module):
    def __init__(self, embedding_matrix, num_heads=16, head_dim=16, dropout=0.2):
        super().__init__()
        news_dim          = num_heads * head_dim
        self.news_encoder = NewsEncoder(embedding_matrix, num_heads, head_dim, dropout)
        self.user_encoder = UserEncoder(news_dim, num_heads, dropout)

    def forward(self, history_ids, candidate_ids, hist_mask=None):
        # history_ids   : (B, H, T)
        # candidate_ids : (B, C, T)
        B, H, T = history_ids.shape
        _, C, _ = candidate_ids.shape

        # Encode history
        hist_vecs = self.news_encoder(
            history_ids.view(-1, T)).view(B, H, -1)         # (B, H, D)

        # Encode candidates (shared encoder)
        cand_vecs = self.news_encoder(
            candidate_ids.view(-1, T)).view(B, C, -1)       # (B, C, D)

        # User representation
        user_vec  = self.user_encoder(hist_vecs, hist_mask) # (B, D)

        # Dot-product click scores
        scores = torch.bmm(
            cand_vecs, user_vec.unsqueeze(-1)).squeeze(-1)  # (B, C)
        return scores


print('Model classes defined.')

## 4. Evaluation Metrics

In [ ]:
def dcg_score(y_true, y_score, k=10):
    order  = np.argsort(y_score)[::-1][:k]
    gains  = np.array(y_true)[order]
    disc   = np.log2(np.arange(len(gains)) + 2)
    return float(np.sum(gains / disc))

def ndcg_score(y_true, y_score, k=10):
    best = dcg_score(y_true, y_true, k)
    return dcg_score(y_true, y_score, k) / best if best > 0 else 0.0

def mrr_score(y_true, y_score):
    order    = np.argsort(y_score)[::-1]
    y_sorted = np.array(y_true)[order]
    for i, v in enumerate(y_sorted):
        if v == 1:
            return 1.0 / (i + 1)
    return 0.0


def evaluate(model, dev_samples, device, batch_size=256):
    """
    Evaluate on dev set.
    dev_samples have variable-length candidate lists, so we process individually.
    For speed we batch the history+candidates encoding where possible.
    """
    model.eval()
    aucs, mrrs, ndcg5s, ndcg10s = [], [], [], []

    with torch.no_grad():
        for s in dev_samples:
            y_true = s['labels']
            if sum(y_true) == 0 or sum(y_true) == len(y_true):
                continue

            hist = torch.tensor(s['history'],    dtype=torch.long).unsqueeze(0).to(device)
            mask = torch.tensor(s['hist_mask'],  dtype=torch.long).unsqueeze(0).to(device)
            cand = torch.tensor(np.array(s['candidates']), dtype=torch.long).unsqueeze(0).to(device)

            scores  = model(hist, cand, mask).squeeze(0).cpu().numpy()
            y_score = scores[:len(y_true)]

            aucs.append(roc_auc_score(y_true, y_score))
            mrrs.append(mrr_score(y_true, y_score))
            ndcg5s.append(ndcg_score(y_true, y_score, 5))
            ndcg10s.append(ndcg_score(y_true, y_score, 10))

    results = {
        'AUC'    : float(np.mean(aucs)),
        'MRR'    : float(np.mean(mrrs)),
        'nDCG@5' : float(np.mean(ndcg5s)),
        'nDCG@10': float(np.mean(ndcg10s)),
    }
    for k, v in results.items():
        print(f'  {k:8s}: {v:.4f}')
    return results

print('Metrics functions defined.')

## 5. Training Function

In [ ]:
def train_model(config, embedding_matrix, train_samples, dev_samples,
                device, run_name='default'):
    """
    Train a NRMS model with the given config dict.
    Returns (model, history_dict)
    """
    loader = DataLoader(
        MINDTrainDataset(train_samples),
        batch_size=config['batch_size'],
        shuffle=True,
        num_workers=0,   # set > 0 on Linux with GPU for speed
        pin_memory=(device.type == 'cuda'),
    )

    model = NRMSModel(
        embedding_matrix,
        num_heads=config['num_heads'],
        head_dim=config['head_dim'],
        dropout=config['dropout'],
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'val_metrics': []}
    best_auc = 0.0

    for epoch in range(config['epochs']):
        model.train()
        total_loss = 0.0
        t0 = time.time()

        for batch in loader:
            hist = batch['history'].to(device)
            cand = batch['candidates'].to(device)
            mask = batch['hist_mask'].to(device)
            tgt  = batch['label'].to(device)

            optimizer.zero_grad()
            scores = model(hist, cand, mask)
            loss   = criterion(scores, tgt)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        history['train_loss'].append(avg_loss)
        scheduler.step()

        elapsed = time.time() - t0
        print(f'[{run_name}] Epoch {epoch+1}/{config["epochs"]}  '
              f'Loss: {avg_loss:.4f}  ({elapsed:.0f}s)')

        # Evaluate every epoch
        print(f'  Evaluating on dev set...')
        metrics = evaluate(model, dev_samples, device)
        history['val_metrics'].append(metrics)

        if metrics['AUC'] > best_auc:
            best_auc = metrics['AUC']
            torch.save(model.state_dict(), f'models/{run_name}_best.pt')
            print(f'  ✓ New best AUC={best_auc:.4f} — checkpoint saved.')

    return model, history


print('train_model() defined.')

## 6. Experiment 1 — Baseline Run

In [ ]:
config_baseline = {
    'batch_size': 64,
    'lr'        : 1e-4,
    'epochs'    : 5,
    'num_heads' : 16,
    'head_dim'  : 16,
    'dropout'   : 0.2,
}

model_baseline, hist_baseline = train_model(
    config_baseline, embedding_matrix,
    train_samples, dev_samples,
    DEVICE, run_name='baseline'
)

In [ ]:
def plot_training_history(history, title='Training Loss & Validation AUC'):
    epochs = range(1, len(history['train_loss']) + 1)
    aucs   = [m['AUC'] for m in history['val_metrics']]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(epochs, history['train_loss'], 'o-', color='steelblue')
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Cross-Entropy Loss')

    axes[1].plot(epochs, aucs, 'o-', color='coral')
    axes[1].set_title('Validation AUC')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('AUC')
    axes[1].set_ylim(0.5, 0.75)

    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.savefig(f'results/{title.replace(" ", "_")}.png', bbox_inches='tight')
    plt.show()

plot_training_history(hist_baseline, 'Baseline Training')

## 7. Experiment 2 — Higher Learning Rate

In [ ]:
config_lr = dict(config_baseline)
config_lr['lr'] = 5e-4

model_lr, hist_lr = train_model(
    config_lr, embedding_matrix,
    train_samples, dev_samples,
    DEVICE, run_name='exp_lr5e4'
)
plot_training_history(hist_lr, 'Experiment LR=5e-4')

## 8. Experiment 3 — Lower Dropout

In [ ]:
config_dropout = dict(config_baseline)
config_dropout['dropout'] = 0.1

model_drop, hist_drop = train_model(
    config_dropout, embedding_matrix,
    train_samples, dev_samples,
    DEVICE, run_name='exp_drop01'
)
plot_training_history(hist_drop, 'Experiment Dropout=0.1')

## 9. Experiment Comparison

In [ ]:
experiments = [
    ('Baseline (lr=1e-4, drop=0.2)', hist_baseline),
    ('LR=5e-4', hist_lr),
    ('Dropout=0.1', hist_drop),
]

print(f'{'Experiment':<35} {'AUC':>6} {'MRR':>6} {'nDCG@5':>8} {'nDCG@10':>9}')
print('-' * 70)
for name, hist in experiments:
    best = max(hist['val_metrics'], key=lambda m: m['AUC'])
    print(f'{name:<35} {best["AUC"]:>6.4f} {best["MRR"]:>6.4f} '
          f'{best["nDCG@5"]:>8.4f} {best["nDCG@10"]:>9.4f}')

## 10. Final Evaluation — Best Checkpoint

In [ ]:
# Load the best saved checkpoint (change run_name if a different experiment won)
best_model = NRMSModel(
    embedding_matrix,
    num_heads=config_baseline['num_heads'],
    head_dim=config_baseline['head_dim'],
    dropout=config_baseline['dropout'],
).to(DEVICE)

best_model.load_state_dict(torch.load('models/baseline_best.pt', map_location=DEVICE))
print('Best checkpoint loaded. Final evaluation:')
final_metrics = evaluate(best_model, dev_samples, DEVICE)

## 11. Error Analysis

In [ ]:
import pandas as pd

NEWS_COLS = ['news_id', 'category', 'subcategory', 'title',
             'abstract', 'url', 'title_entities', 'abstract_entities']

news_df = pd.read_csv('data/MINDsmall_dev/news.tsv', sep='\t', names=NEWS_COLS)
news_df_train = pd.read_csv('data/MINDsmall_train/news.tsv', sep='\t', names=NEWS_COLS)
news_all = pd.concat([news_df, news_df_train]).drop_duplicates('news_id')
news_meta = news_all.set_index('news_id')[['category', 'title']].to_dict(orient='index')

with open('data/processed/tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

with open('data/processed/news_encoded.pkl', 'rb') as f:
    news_encoded = pickle.load(f)

# Collect per-impression AUC
BEH_COLS = ['impression_id', 'user_id', 'time', 'history', 'impressions']
beh_dev  = pd.read_csv('data/MINDsmall_dev/behaviors.tsv', sep='\t', names=BEH_COLS)

best_model.eval()
per_imp = []

for i, (s, (_, row)) in enumerate(zip(dev_samples, beh_dev.iterrows())):
    y_true = s['labels']
    if sum(y_true) == 0 or sum(y_true) == len(y_true):
        continue

    with torch.no_grad():
        hist = torch.tensor(s['history'],   dtype=torch.long).unsqueeze(0).to(DEVICE)
        mask = torch.tensor(s['hist_mask'], dtype=torch.long).unsqueeze(0).to(DEVICE)
        cand = torch.tensor(np.array(s['candidates']), dtype=torch.long).unsqueeze(0).to(DEVICE)
        scores = best_model(hist, cand, mask).squeeze(0).cpu().numpy()

    y_score   = scores[:len(y_true)]
    hist_len  = int(s['hist_mask'].sum())
    imp_auc   = roc_auc_score(y_true, y_score)
    per_imp.append({'auc': imp_auc, 'hist_len': hist_len,
                    'n_cands': len(y_true)})

    if i >= 2000:  # limit for speed
        break

err_df = pd.DataFrame(per_imp)
print('Per-impression AUC stats:')
print(err_df['auc'].describe().round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# AUC distribution
axes[0].hist(err_df['auc'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(err_df['auc'].mean(), color='red', linestyle='--',
                label=f'Mean={err_df["auc"].mean():.3f}')
axes[0].set_title('Per-Impression AUC Distribution')
axes[0].set_xlabel('AUC')
axes[0].set_ylabel('Count')
axes[0].legend()

# AUC vs history length (binned)
err_df['hist_bin'] = pd.cut(err_df['hist_len'],
    bins=[0, 5, 15, 30, 50], labels=['0-5','6-15','16-30','31-50'])
auc_by_hist = err_df.groupby('hist_bin', observed=True)['auc'].mean()
axes[1].bar(auc_by_hist.index.astype(str), auc_by_hist.values,
            color='coral', edgecolor='white')
axes[1].set_title('Mean AUC by History Length')
axes[1].set_xlabel('History Length')
axes[1].set_ylabel('Mean AUC')
axes[1].set_ylim(0.5, 0.75)

plt.tight_layout()
plt.savefig('results/error_analysis.png', bbox_inches='tight')
plt.show()
print('\nInsight: Users with longer history tend to get better recommendations.')
print('Cold-start (hist_len 0-5) is the hardest case.')